# Cross-Industry Accelerator — Create Data Agents
### Auto-Create Fabric IQ QA and Operations Agents

This notebook creates **two agents** backed by the ontology and data sources:

| Agent | Purpose | Power |
|---|---|---|
| **QA Agent** | Answer ad-hoc data questions in natural language | Queries Lakehouse + Eventhouse via ontology |
| **Ops Agent** | Monitor operations and surface alerts/anomalies | Real-time event analysis via ontology |

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Resolves the ontology item ID from the workspace
3. Creates the QA Agent item linked to the ontology
4. Creates the Ops Agent item linked to the ontology
5. Configures each agent with industry-appropriate instructions

> **Prerequisites:**
> 1. Run `05_Create_Ontology.ipynb` to create the ontology first
> 2. Run `04_Create_Semantic_Model.ipynb` to create the semantic model
> 3. Data must be loaded (Lakehouse + Eventhouse) for agents to query
> 4. Each industry must have a `{Industry}_Agent_Instructions.ipynb` notebook that defines `QA_AGENT_INSTRUCTIONS` and `OPS_AGENT_INSTRUCTIONS`. If the notebook is missing, generic instructions are used.

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD INDUSTRY-SPECIFIC AGENT INSTRUCTIONS
# ════════════════════════════════════════════════════════════════════════
# Calls {INDUSTRY_LABEL}_Agent_Instructions via notebookutils.notebook.run().
# The child notebook returns instructions as JSON via notebook.exit().
# Falls back to generic instructions if the notebook is missing.

import json

_instructions_nb = f"{INDUSTRY_LABEL}_Agent_Instructions"
print(f"Loading agent instructions from: {_instructions_nb}")

try:
    _result = notebookutils.notebook.run(_instructions_nb, 600)
    _parsed = json.loads(_result)
    QA_AGENT_INSTRUCTIONS = _parsed["qa"]
    OPS_AGENT_INSTRUCTIONS = _parsed["ops"]
    QA_FEWSHOTS = _parsed.get("qa_fewshots", {})
    OPS_FEWSHOTS = _parsed.get("ops_fewshots", {})
    QA_DS_INSTRUCTIONS = _parsed.get("qa_ds_instructions", {})
    OPS_DS_INSTRUCTIONS = _parsed.get("ops_ds_instructions", {})
    print(f"  QA instructions:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
    print(f"  Ops instructions: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
    print(f"  QA fewshots:  {sum(len(v) for v in QA_FEWSHOTS.values())} queries across {len(QA_FEWSHOTS)} datasources")
    print(f"  Ops fewshots: {sum(len(v) for v in OPS_FEWSHOTS.values())} queries across {len(OPS_FEWSHOTS)} datasources")
except Exception as e:
    print(f"  Could not load {_instructions_nb}: {e}")
    print(f"  Will use generic instructions.")
    QA_AGENT_INSTRUCTIONS = None
    OPS_AGENT_INSTRUCTIONS = None
    QA_FEWSHOTS = {}
    OPS_FEWSHOTS = {}
    QA_DS_INSTRUCTIONS = {}
    OPS_DS_INSTRUCTIONS = {}

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RESOLVE WORKSPACE ITEMS
# ════════════════════════════════════════════════════════════════════════

import sempy.fabric as fabric
import json, requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

# Resolve Ontology item ID
ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
if ont_matches.empty:
    print(f"⚠ Ontology '{ONTOLOGY_NAME}' not found. Run 05_Create_Ontology first.")
    print(f"Available ontologies:")
    print(items_df[items_df["Type"] == "Ontology"][["Display Name", "Id"]].to_string(index=False))
    ontology_item_id = None
else:
    ontology_item_id = str(ont_matches.iloc[0].Id)
    print(f"✓ Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

# Resolve Lakehouse item ID
lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"✓ Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

# Resolve Eventhouse item ID
eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    if not eh_matches.empty:
        eventhouse_item_id = str(eh_matches.iloc[0].Id)
        print(f"✓ Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

print(f"\n  QA Agent name:  {DATA_AGENT_NAME}")
print(f"  Ops Agent name: {OPS_AGENT_NAME}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# INSTALL FABRIC DATA AGENT SDK (requires kernel restart after install)
# ════════════════════════════════════════════════════════════════════════
# ⚡ RUN THIS CELL ONCE, then:
#    1. Restart the kernel (Session → Restart session)
#    2. SKIP this cell (do NOT re-run it)
#    3. Run ALL remaining cells from the next cell onward
#       (Select next cell → Shift+Enter through each, or Run All Below)
#
# The next cell (Import SDK) re-loads config, workspace items, and
# instructions automatically — no need to re-run earlier cells.
# ════════════════════════════════════════════════════════════════════════

import subprocess, sys, os

_AGENT_LIB = "/tmp/agent_sdk_lib"
os.makedirs(_AGENT_LIB, exist_ok=True)

# Install everything to an isolated path (avoids system typing_extensions conflict)
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    f"--target={_AGENT_LIB}",
    "--upgrade", "--no-cache-dir",
    "fabric-data-agent-sdk", "typing_extensions>=4.13", "requests>=2.32.3"
])

# Verify
_te_path = os.path.join(_AGENT_LIB, "typing_extensions.py")
with open(_te_path) as f:
    assert "Sentinel" in f.read(), "Sentinel not found in installed typing_extensions!"
print(f"  ✓ typing_extensions at {_te_path} has Sentinel")

print("\n" + "═" * 60)
print("  ✓ SDK INSTALLED SUCCESSFULLY")
print("═" * 60)
print()
print("  NEXT STEPS:")
print("  1. Restart the kernel  (Session → Restart session)")
print("  2. SKIP this cell      (do NOT re-run it)")
print("  3. Run all cells below (the next cell reloads everything)")
print("═" * 60)

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# IMPORT SDK + RELOAD ALL STATE (run after kernel restart)
# ════════════════════════════════════════════════════════════════════════
# This cell re-establishes everything lost during kernel restart:
#   - Industry config variables (INDUSTRY, WAREHOUSE_NAME, etc.)
#   - Agent instructions + fewshots from the industry notebook
#   - Workspace item resolution (ontology, lakehouse, eventhouse IDs)
#   - SDK imports
# ════════════════════════════════════════════════════════════════════════

import sys, json

# 1. Prepend isolated SDK path
_AGENT_LIB = "/tmp/agent_sdk_lib"
if _AGENT_LIB not in sys.path:
    sys.path.insert(0, _AGENT_LIB)

import typing_extensions as _te
print(f"  typing_extensions: {_te.__file__} (Sentinel={hasattr(_te, 'Sentinel')})")

# 2. Reload industry config (lost after kernel restart)
print("\n── Reloading industry config...")
exec(open("/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/notebookutils/__init__.py").read()) if False else None
%run 00_Industry_Config

# 3. Reload agent instructions
print("\n── Loading agent instructions...")
_instructions_nb = f"{INDUSTRY_LABEL}_Agent_Instructions"
try:
    _result = notebookutils.notebook.run(_instructions_nb, 600)
    _parsed = json.loads(_result)
    QA_AGENT_INSTRUCTIONS = _parsed["qa"]
    OPS_AGENT_INSTRUCTIONS = _parsed["ops"]
    QA_FEWSHOTS = _parsed.get("qa_fewshots", {})
    OPS_FEWSHOTS = _parsed.get("ops_fewshots", {})
    QA_DS_INSTRUCTIONS = _parsed.get("qa_ds_instructions", {})
    OPS_DS_INSTRUCTIONS = _parsed.get("ops_ds_instructions", {})
    print(f"  QA instructions:  {len(QA_AGENT_INSTRUCTIONS):,} chars")
    print(f"  Ops instructions: {len(OPS_AGENT_INSTRUCTIONS):,} chars")
    print(f"  QA fewshots:  {sum(len(v) for v in QA_FEWSHOTS.values())} queries")
    print(f"  Ops fewshots: {sum(len(v) for v in OPS_FEWSHOTS.values())} queries")
    print(f"  QA DS instructions: {len(QA_DS_INSTRUCTIONS)} datasources")
    print(f"  Ops DS instructions: {len(OPS_DS_INSTRUCTIONS)} datasources")
except Exception as e:
    print(f"  Could not load {_instructions_nb}: {e}")
    QA_AGENT_INSTRUCTIONS = OPS_AGENT_INSTRUCTIONS = None
    QA_FEWSHOTS = OPS_FEWSHOTS = QA_DS_INSTRUCTIONS = OPS_DS_INSTRUCTIONS = {}

# 4. Resolve workspace items
print("\n── Resolving workspace items...")
import sempy.fabric as fabric
import requests

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
items_df = fabric.list_items()

ont_matches = items_df[(items_df["Type"] == "Ontology") & (items_df["Display Name"] == ONTOLOGY_NAME)]
ontology_item_id = str(ont_matches.iloc[0].Id) if not ont_matches.empty else None
print(f"  Ontology: {ONTOLOGY_NAME} → {ontology_item_id}")

lh_matches = items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == LAKEHOUSE_NAME)]
lakehouse_item_id = str(lh_matches.iloc[0].Id) if not lh_matches.empty else None
print(f"  Lakehouse: {LAKEHOUSE_NAME} → {lakehouse_item_id}")

eventhouse_item_id = None
if EVENTHOUSE_NAME:
    eh_matches = items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == EVENTHOUSE_NAME)]
    eventhouse_item_id = str(eh_matches.iloc[0].Id) if not eh_matches.empty else None
    print(f"  Eventhouse: {EVENTHOUSE_NAME} → {eventhouse_item_id}")

# 5. Import SDK
print("\n── Importing fabric-data-agent-sdk...")
from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print(f"\n✓ All state restored. Ready to create agents.")
print(f"  QA Agent:  {DATA_AGENT_NAME}")
print(f"  Ops Agent: {OPS_AGENT_NAME}")

from fabric.dataagent.client import (
    FabricDataAgentManagement,
    create_data_agent,
    delete_data_agent,
)

print("fabric-data-agent-sdk imported successfully.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE QA AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

import time, requests as req_lib
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
api_headers = {"Authorization": f"Bearer {access_token}"}
BASE = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# ── Helper: create agent with retry (name may not be available immediately after delete)
def _create_agent_with_retry(name, max_retries=5, wait_sec=25):
    """Retry create_data_agent if the name is temporarily unavailable after deletion."""
    for attempt in range(1, max_retries + 1):
        try:
            agent = create_data_agent(name)
            return agent
        except Exception as e:
            err = str(e).lower()
            if "not available yet" in err or "notavailableyet" in err:
                print(f"    Name not available yet, retrying in {wait_sec}s (attempt {attempt}/{max_retries})...")
                time.sleep(wait_sec)
            elif "already in use" in err:
                print(f"    Already exists. Connecting to existing...")
                return FabricDataAgentManagement(name)
            else:
                raise
    raise RuntimeError(f"Agent name '{name}' still not available after {max_retries} retries.")

# ── Clean up existing agents
items_df = fabric.list_items()
for agent_name in [DATA_AGENT_NAME, OPS_AGENT_NAME]:
    deleted = False
    try:
        delete_data_agent(agent_name)
        print(f"  Deleted existing agent '{agent_name}' via SDK.")
        deleted = True
    except Exception:
        pass
    if not deleted:
        matches = items_df[items_df["Display Name"] == agent_name]
        if not matches.empty:
            for _, row in matches.iterrows():
                item_id = str(row["Id"])
                dr = req_lib.delete(f"{BASE}/items/{item_id}", headers=api_headers, timeout=30)
                print(f"  Deleted '{agent_name}' via REST: HTTP {dr.status_code}")
                deleted = True
    if not deleted:
        print(f"  '{agent_name}' — no existing agent found (clean slate).")

# Wait for names to become available after deletion
print("  Waiting for name availability...")
time.sleep(25)

# ── Create QA Agent
print(f"\nCreating QA Agent: {DATA_AGENT_NAME}...")
qa_agent = None
try:
    qa_agent = _create_agent_with_retry(DATA_AGENT_NAME)
    print(f"  Created: {DATA_AGENT_NAME}")
except Exception as e:
    print(f"  Failed to create QA agent: {e}")

if qa_agent:
    # Set instructions
    if 'QA_AGENT_INSTRUCTIONS' in dir() and QA_AGENT_INSTRUCTIONS:
        qa_instructions = QA_AGENT_INSTRUCTIONS
    else:
        qa_instructions = (
            f"You are a data analyst agent for the {INDUSTRY} industry. "
            f"Answer questions about {INDUSTRY} data using the connected data sources. "
            f"Query dimension tables (dim_*) for reference data and fact tables (fact_*) for metrics. "
            f"Use the Warehouse for SQL queries, the KQL Database for real-time event data, "
            f"and the Semantic Model for pre-built measures. "
            f"You are scoped to {INDUSTRY} data only."
        )
    qa_agent.update_configuration(instructions=qa_instructions)
    print(f"  Instructions set ({len(qa_instructions)} chars).")

    # Add data sources
    for ds_name, ds_type in [
        (WAREHOUSE_NAME, "warehouse"),
        (SEMANTIC_MODEL_NAME, "semanticmodel"),
        (LAKEHOUSE_NAME, "lakehouse"),
    ]:
        print(f"  Adding {ds_type}: {ds_name}...")
        try:
            qa_agent.add_datasource(ds_name, type=ds_type)
            print(f"    Added.")
        except Exception as e:
            print(f"    {e}")

    if EVENTHOUSE_DATABASE:
        print(f"  Adding kqldatabase: {EVENTHOUSE_DATABASE}...")
        try:
            qa_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"    Added.")
        except Exception as e:
            print(f"    {e}")

    # Upload per-datasource instructions and example queries (fewshots)
    # Per-datasource instructions describe each data source to the agent
    if 'QA_DS_INSTRUCTIONS' in dir() and QA_DS_INSTRUCTIONS:
        for ds_name, ds_instruction in QA_DS_INSTRUCTIONS.items():
            if ds_instruction:
                print(f"  Setting instruction for {ds_name}...")
                try:
                    qa_agent.update_configuration(
                        datasource_name=ds_name,
                        description=ds_instruction
                    )
                    print(f"    Set.")
                except Exception as e:
                    print(f"    DS instruction: {e}")

    if 'QA_FEWSHOTS' in dir() and QA_FEWSHOTS:
        for ds_name, fewshots in QA_FEWSHOTS.items():
            if fewshots:
                print(f"  Uploading {len(fewshots)} example queries for {ds_name}...")
                try:
                    qa_agent.upload_fewshots(ds_name, fewshots)
                    print(f"    Uploaded.")
                except Exception as e:
                    print(f"    Fewshots: {e}")
    else:
        print(f"  No per-datasource example queries (QA_FEWSHOTS not set).")

    # Publish
    print(f"  Publishing...")
    try:
        qa_agent.publish()
        print(f"  Published: {DATA_AGENT_NAME}")
    except Exception as e:
        print(f"  Publish: {e}")

    try:
        print(f"  Config: {qa_agent.get_configuration()}")
        print(f"  Sources: {qa_agent.get_datasources()}")
    except Exception:
        pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE OPS AGENT + ADD DATA SOURCES (via fabric-data-agent-sdk)
# ════════════════════════════════════════════════════════════════════════

print(f"\nCreating Ops Agent: {OPS_AGENT_NAME}...")
ops_agent = None
try:
    ops_agent = _create_agent_with_retry(OPS_AGENT_NAME)
    print(f"  Created: {OPS_AGENT_NAME}")
except Exception as e:
    print(f"  Failed to create Ops agent: {e}")

if ops_agent:
    # Set instructions
    if 'OPS_AGENT_INSTRUCTIONS' in dir() and OPS_AGENT_INSTRUCTIONS:
        ops_instructions = OPS_AGENT_INSTRUCTIONS
    else:
        ops_instructions = (
            f"You are an operations monitoring agent for the {INDUSTRY} industry. "
            f"Analyze event streams and fact tables to detect anomalies and surface alerts. "
            f"Focus on KPIs, SLA compliance, quality metrics, and risk indicators. "
            f"Use the Warehouse for historical trends and the KQL Database for real-time events. "
            f"When reporting issues, provide severity and recommended actions. "
            f"You are scoped to {INDUSTRY} operational data only."
        )
    ops_agent.update_configuration(instructions=ops_instructions)
    print(f"  Instructions set ({len(ops_instructions)} chars).")

    # Add data sources
    print(f"  Adding Warehouse: {WAREHOUSE_NAME}...")
    try:
        ops_agent.add_datasource(WAREHOUSE_NAME, type="warehouse")
        print(f"    Warehouse added.")
    except Exception as e:
        print(f"    Warehouse: {e}")

    if EVENTHOUSE_DATABASE:
        print(f"  Adding KQL Database: {EVENTHOUSE_DATABASE}...")
        try:
            ops_agent.add_datasource(EVENTHOUSE_DATABASE, type="kqldatabase")
            print(f"    KQL Database added.")
        except Exception as e:
            print(f"    KQL DB: {e}")

    # Upload per-datasource instructions and example queries (fewshots)
    if 'OPS_DS_INSTRUCTIONS' in dir() and OPS_DS_INSTRUCTIONS:
        for ds_name, ds_instruction in OPS_DS_INSTRUCTIONS.items():
            if ds_instruction:
                print(f"  Setting instruction for {ds_name}...")
                try:
                    ops_agent.update_configuration(
                        datasource_name=ds_name,
                        description=ds_instruction
                    )
                    print(f"    Set.")
                except Exception as e:
                    print(f"    DS instruction: {e}")

    if 'OPS_FEWSHOTS' in dir() and OPS_FEWSHOTS:
        for ds_name, fewshots in OPS_FEWSHOTS.items():
            if fewshots:
                print(f"  Uploading {len(fewshots)} example queries for {ds_name}...")
                try:
                    ops_agent.upload_fewshots(ds_name, fewshots)
                    print(f"    Uploaded.")
                except Exception as e:
                    print(f"    Fewshots: {e}")
    else:
        print(f"  No per-datasource example queries (OPS_FEWSHOTS not set).")

    # Publish
    print(f"  Publishing...")
    try:
        ops_agent.publish()
        print(f"  Ops Agent published: {OPS_AGENT_NAME}")
    except Exception as e:
        print(f"  Publish: {e}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# AGENT CREATION SUMMARY
# ════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*60}")
print(f"DATA AGENT SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*60}")
print(f"")
print(f"  Ontology:       {ONTOLOGY_NAME}")
print(f"  Lakehouse:      {LAKEHOUSE_NAME}")
print(f"  Eventhouse:     {EVENTHOUSE_NAME or 'N/A'}")
print(f"")
print(f"  QA Agent:       {DATA_AGENT_NAME}")
print(f"    → Answers ad-hoc data questions about {INDUSTRY} data")
print(f"    → Queries: dimensions, facts, aggregates")
print(f"")
print(f"  Ops Agent:      {OPS_AGENT_NAME}")
print(f"    → Monitors real-time events and operational metrics")
print(f"    → Surfaces: anomalies, SLA breaches, quality alerts")
print(f"")
print(f"✅ Agent creation complete.")
print(f"   Next: Run 07_Create_Dashboards.ipynb to create real-time and Power BI dashboards.")